In [6]:
import os
import glob
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from PIL import Image, ImageTk
from io import BytesIO
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2 import preprocess_input as res_pre
import time
import threading
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox, filedialog
from datetime import datetime
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import queue
import gc
import atexit
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import multiprocessing as mp
import psutil
from collections import defaultdict, deque
import weakref
warnings.filterwarnings('ignore')

# 기본 설정
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# 성능 프로파일링 도구
class PerformanceProfiler:
    def __init__(self):
        self.start_time = None
        self.checkpoints = []
        
    def start(self):
        self.start_time = time.time()
        self.checkpoints = []
        
    def checkpoint(self, name):
        if self.start_time:
            elapsed = time.time() - self.start_time
            self.checkpoints.append((name, elapsed))
            
    def summary(self):
        if not self.checkpoints:
            return
            
        total_time = self.checkpoints[-1][1]
        for i, (name, elapsed) in enumerate(self.checkpoints):
            if i > 0:
                step_time = elapsed - self.checkpoints[i-1][1]
                print(f"  [PERF] {name}: {step_time:.2f}초")

# 메모리 효율적 LRU 캐시
class MemoryEfficientCache:
    def __init__(self, max_size=1000, max_memory_mb=500):
        self.max_size = max_size
        self.max_memory_bytes = max_memory_mb * 1024 * 1024
        self.cache = {}
        self.access_order = deque()
        self.current_memory = 0
        
    def get(self, key):
        if key in self.cache:
            # LRU 업데이트
            self.access_order.remove(key)
            self.access_order.append(key)
            return self.cache[key]
        return None
    
    def put(self, key, value):
        # 메모리 사용량 추정 (이미지 기준)
        if hasattr(value[0], 'size') and value[0] is not None:
            estimated_size = value[0].size[0] * value[0].size[1] * 3  # RGB
        else:
            estimated_size = 1024  # 기본값
            
        # 메모리 한계 체크
        while (self.current_memory + estimated_size > self.max_memory_bytes or 
               len(self.cache) >= self.max_size) and self.access_order:
            self._evict_oldest()
        
        # 새 항목 추가
        if key not in self.cache:
            self.cache[key] = value
            self.access_order.append(key)
            self.current_memory += estimated_size
    
    def _evict_oldest(self):
        if self.access_order:
            oldest_key = self.access_order.popleft()
            if oldest_key in self.cache:
                old_value = self.cache.pop(oldest_key)
                # 메모리 사용량 업데이트
                if hasattr(old_value[0], 'size') and old_value[0] is not None:
                    self.current_memory -= old_value[0].size[0] * old_value[0].size[1] * 3
                else:
                    self.current_memory -= 1024

# 상세한 메모리 사용량 모니터링
def monitor_memory_detailed():
    """상세한 메모리 사용량 모니터링"""
    try:
        memory = psutil.virtual_memory()
        print(f"[MEMORY] 물리 메모리: {memory.used/(1024**3):.1f}GB / {memory.total/(1024**3):.1f}GB ({memory.percent}%)")
        print(f"[MEMORY] 사용 가능: {memory.available/(1024**3):.1f}GB")
        
        # GPU 메모리 모니터링
        try:
            gpu_devices = tf.config.experimental.list_physical_devices('GPU')
            if gpu_devices:
                try:
                    memory_info = tf.config.experimental.get_memory_info('GPU:0')
                    current_gb = memory_info['current'] / (1024**3)
                    peak_gb = memory_info['peak'] / (1024**3)
                    print(f"[GPU MEMORY] 현재: {current_gb:.1f}GB, 최대: {peak_gb:.1f}GB")
                except:
                    pass
        except Exception as e:
            print(f"[GPU MEMORY] GPU 메모리 정보 조회 실패: {e}")
        
        # 프로세스별 메모리 사용량
        process = psutil.Process()
        process_memory = process.memory_info()
        print(f"[PROCESS MEMORY] 현재 프로세스: {process_memory.rss/(1024**3):.1f}GB RSS")
                
    except Exception as e:
        print(f"[MEMORY] 메모리 모니터링 오류: {e}")

# 시스템 리소스 감지 및 최적 설정 계산
def auto_optimize_settings():
    """시스템 리소스 기반 최적화"""
    try:
        cpu_count = mp.cpu_count()
        memory_gb = psutil.virtual_memory().total / (1024**3)
        
        # GPU 감지
        try:
            gpus = tf.config.experimental.list_physical_devices('GPU')
            gpu_count = len(gpus)
        except:
            gpu_count = 0
        
        # 메모리 기반 동적 배치 크기 계산
        if gpu_count > 0:
            # GPU 환경: 메모리에 따라 배치 크기 조정
            if memory_gb >= 32:
                batch_size = min(64, int(memory_gb * 1.5))
            elif memory_gb >= 16:
                batch_size = min(32, int(memory_gb * 1.2))
            else:
                batch_size = min(16, max(8, int(memory_gb)))
            
            max_workers = min(cpu_count * 2, 48)
            download_workers = min(cpu_count, 20)
        else:
            # CPU 전용: 보수적 설정
            batch_size = min(16, max(4, cpu_count // 2))
            max_workers = min(cpu_count, 16)
            download_workers = min(cpu_count // 2, 8)
        
        print(f"[OPTIMIZE] CPU: {cpu_count}코어, RAM: {memory_gb:.1f}GB, GPU: {gpu_count}개")
        print(f"[OPTIMIZE] 최적화 설정 - 배치: {batch_size}, 워커: {max_workers}, 다운로드: {download_workers}")
        print(f"[OPTIMIZE] 예상 메모리 사용량: ~{batch_size * 0.25:.1f}GB")
        
        return batch_size, max_workers, download_workers, gpu_count > 0
        
    except Exception as e:
        print(f"[OPTIMIZE] 최적화 실패, 안전값 사용: {e}")
        return 16, 8, 4, False

BATCH_SIZE, MAX_WORKERS, DOWNLOAD_WORKERS, AUTO_GPU_DETECTED = auto_optimize_settings()

# GPU 최적화 설정
def setup_gpu():
    """GPU 설정 및 최적화"""
    try:
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            try:
                # 메모리 증가 허용 (필수)
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                
                # 혼합 정밀도 활성화 (성능 향상)
                try:
                    policy = tf.keras.mixed_precision.Policy('mixed_float16')
                    tf.keras.mixed_precision.set_global_policy(policy)
                    print(f"[GPU OPTIMIZE] 혼합정밀도 활성화")
                except Exception as e:
                    print(f"[GPU OPTIMIZE] 혼합정밀도 비활성화: {e}")
                
                # XLA 최적화 시도 (안전하게)
                try:
                    tf.config.optimizer.set_jit(True)
                    print(f"[GPU OPTIMIZE] XLA 컴파일 활성화")
                except Exception as e:
                    print(f"[GPU OPTIMIZE] XLA 비활성화: {e}")
                except AttributeError:
                    # 구버전 TensorFlow에서는 이 기능이 없을 수 있음
                    print(f"[GPU OPTIMIZE] XLA 기능 없음 (TensorFlow 버전 이슈)")
                
                tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
                print(f"[GPU OPTIMIZE] GPU 최적화 완료: {len(gpus)}개")
                return True
                
            except RuntimeError as e:
                print(f"[GPU WARNING] GPU 설정 실패: {e}")
                return False
        else:
            print("[GPU INFO] GPU를 찾을 수 없습니다. CPU를 사용합니다.")
            return False
    except Exception as e:
        print(f"[GPU ERROR] GPU 초기화 오류: {e}")
        return False

GPU_AVAILABLE = setup_gpu()
IMG_SIZE = (224, 224)

class OptimizedDownloader:
    """메모리 및 네트워크 최적화된 이미지 다운로더"""
    
    def __init__(self):
        self.session_pool = self._create_optimized_session_pool()
        self.download_cache = MemoryEfficientCache(max_size=2000, max_memory_mb=300)
        self.failed_urls = set()
        self.session_index = 0
        
        # 성능 통계
        self.cache_hits = 0
        self.cache_misses = 0
        
    def _create_optimized_session_pool(self):
        """최적화된 세션 풀 생성"""
        sessions = []
        pool_size = min(16, DOWNLOAD_WORKERS) or 4
        
        for _ in range(pool_size):
            session = requests.Session()
            
            # 더 적극적인 재시도 정책
            retry_strategy = Retry(
                total=3,
                backoff_factor=0.1,
                status_forcelist=[429, 500, 502, 503, 504]
            )
            
            adapter = HTTPAdapter(
                pool_connections=100,  # 더 많은 연결 풀
                pool_maxsize=100,
                max_retries=retry_strategy
            )
            
            session.mount("http://", adapter)
            session.mount("https://", adapter)
            
            # 최적화된 헤더
            session.headers.update({
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Accept': 'image/webp,image/avif,image/apng,image/*,*/*;q=0.8',
                'Accept-Encoding': 'gzip, deflate, br',
                'Accept-Language': 'ko-KR,ko;q=0.9,en;q=0.8',
                'Connection': 'keep-alive',
                'Cache-Control': 'max-age=300',
            })
            
            sessions.append(session)
        
        return sessions
    
    def _get_session(self):
        """라운드 로빈으로 세션 선택"""
        session = self.session_pool[self.session_index % len(self.session_pool)]
        self.session_index += 1
        return session
    
    def download_single(self, url):
        """최적화된 단일 다운로드"""
        try:
            if not url or str(url).strip() == '' or pd.isna(url):
                return None, "URL이 비어있습니다"
            
            url = str(url).strip()
            
            # 실패한 URL 빠른 체크
            if url in self.failed_urls:
                return None, "이전 실패 URL"
            
            # 캐시 체크 (메모리 효율적)
            cached_result = self.download_cache.get(url)
            if cached_result:
                self.cache_hits += 1
                return cached_result
            
            self.cache_misses += 1
            
            if not url.startswith(('http://', 'https://')):
                result = (None, "유효하지 않은 URL 프로토콜")
                self.failed_urls.add(url)
                return result
            
            session = self._get_session()
            
            # 더 빠른 타임아웃과 스트리밍
            response = session.get(url, timeout=(2, 4), verify=False, stream=True)
            response.raise_for_status()
            
            content_type = response.headers.get('content-type', '').lower()
            if content_type and not any(img_type in content_type for img_type in ['image/', 'jpeg', 'jpg', 'png', 'gif', 'webp', 'avif']):
                result = (None, "이미지 파일이 아님")
                self.failed_urls.add(url)
                return result
            
            # 메모리 효율적 이미지 로딩
            image_data = BytesIO()
            downloaded = 0
            max_size = 20 * 1024 * 1024  # 20MB
            chunk_size = 32768  # 32KB 청크
            
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    downloaded += len(chunk)
                    if downloaded > max_size:
                        result = (None, "파일이 너무 큼")
                        self.failed_urls.add(url)
                        return result
                    image_data.write(chunk)
            
            image_data.seek(0)
            
            try:
                img = Image.open(image_data)
                img.load()  # 실제 로딩
                
                # 메모리 해제
                image_data.close()
                image_data = None
                
            except Exception as e:
                result = (None, "이미지 파싱 오류")
                self.failed_urls.add(url)
                return result
            
            # 최소 크기 체크
            if img.width < 32 or img.height < 32:
                result = (None, "이미지가 너무 작음")
                self.failed_urls.add(url)
                return result
            
            # RGB 변환 (필요시)
            if img.mode != 'RGB':
                try:
                    img = img.convert('RGB')
                except Exception as e:
                    result = (None, "RGB 변환 실패")
                    self.failed_urls.add(url)
                    return result
            
            result = (img, "성공")
            
            # 캐시에 저장 (메모리 효율적)
            self.download_cache.put(url, result)
            
            return result
            
        except requests.exceptions.Timeout:
            result = (None, "타임아웃")
            self.failed_urls.add(url)
            return result
        except requests.exceptions.ConnectionError:
            result = (None, "연결 오류")
            self.failed_urls.add(url)
            return result
        except requests.exceptions.HTTPError as e:
            result = (None, "HTTP 오류")
            self.failed_urls.add(url)
            return result
        except Exception as e:
            result = (None, "예상치 못한 오류")
            self.failed_urls.add(url)
            return result
        finally:
            # 즉시 메모리 정리
            if 'image_data' in locals() and image_data:
                image_data.close()
    
    def download_batch(self, urls, batch_size=None):
        """최적화된 배치 다운로드"""
        if batch_size is None:
            batch_size = DOWNLOAD_WORKERS
        
        # 벡터화된 캐시 체크
        cached_results = []
        new_urls = []
        
        for url in urls:
            if url in self.failed_urls:
                cached_results.append((url, (None, "이전 실패")))
            else:
                cached_result = self.download_cache.get(url)
                if cached_result:
                    cached_results.append((url, cached_result))
                    self.cache_hits += 1
                else:
                    new_urls.append(url)
                    self.cache_misses += 1
        
        # 새로운 URL만 병렬 다운로드
        new_results = []
        if new_urls:
            with ThreadPoolExecutor(max_workers=batch_size, thread_name_prefix="download") as executor:
                future_to_url = {executor.submit(self.download_single, url): url for url in new_urls}
                
                for future in as_completed(future_to_url, timeout=30):
                    url = future_to_url[future]
                    try:
                        result = future.result(timeout=1)
                        new_results.append((url, result))
                    except Exception as e:
                        new_results.append((url, (None, "다운로드 실패")))
                        self.failed_urls.add(url)
        
        return cached_results + new_results
    
    def get_cache_stats(self):
        """캐시 통계 반환"""
        total_requests = self.cache_hits + self.cache_misses
        hit_rate = (self.cache_hits / total_requests * 100) if total_requests > 0 else 0
        return {
            'hit_rate': hit_rate,
            'cache_size': len(self.download_cache.cache),
            'memory_usage_mb': self.download_cache.current_memory / (1024 * 1024)
        }

class FoodImagePredictor:
    def __init__(self, model_dir='models', timestamp='20250716_144252'):
        self.model_dir = model_dir
        self.timestamp = timestamp
        self.eff_model = None
        self.res_model = None
        self.xgb_model = None
        self.index_to_label = {}
        self.properties_data = {}
        
        self.downloader = OptimizedDownloader()
        self.profiler = PerformanceProfiler()
        
        # GPU 최적화를 위한 텐서 재사용
        self._tensor_cache = {}
        self._feature_models = {}
        
    def log(self, message, level="INFO"):
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] [{level}] {message}")
    
    def load_models(self):
        try:
            self.log("모델 로드 중...", "INFO")
            if GPU_AVAILABLE:
                self.log("GPU 최적화 모드로 모델 로드 중...", "SUCCESS")
                with tf.device('/GPU:0'):
                    self._load_models_on_device()
            else:
                self.log("CPU 최적화 모드로 모델 로드 중...", "WARNING")
                with tf.device('/CPU:0'):
                    self._load_models_on_device()
        except Exception as e:
            self.log(f"모델 로드 실패: {e}", "ERROR")
            raise
    
    def _load_models_on_device(self):
        label_map_path = f"{self.model_dir}/label_to_index_{self.timestamp}.joblib"
        if not os.path.exists(label_map_path):
            raise FileNotFoundError(f"라벨 매핑 파일이 없습니다: {label_map_path}")
        
        label_map = joblib.load(label_map_path)
        self.index_to_label = {v: k for k, v in label_map.items()}
        num_classes = len(label_map)
        self.log(f"라벨 매핑 로드 완료 - 클래스 수: {num_classes}", "SUCCESS")
        
        # EfficientNet 모델 로드 및 최적화
        self.log("EfficientNet 모델 로드 중...", "INFO")
        self.eff_model = self.build_optimized_model(EfficientNetB0, num_classes)
        eff_weights_path = f"{self.model_dir}/effnet_model_best_{self.timestamp}.h5"
        if not os.path.exists(eff_weights_path):
            raise FileNotFoundError(f"EfficientNet 가중치 파일이 없습니다: {eff_weights_path}")
        self.eff_model.load_weights(eff_weights_path)
        self.log("EfficientNet 모델 로드 완료", "SUCCESS")
        
        # ResNet 모델 로드 및 최적화
        self.log("ResNet 모델 로드 중...", "INFO")
        self.res_model = self.build_optimized_model(ResNet50V2, num_classes)
        res_weights_path = f"{self.model_dir}/resnet_model_best_{self.timestamp}.h5"
        if not os.path.exists(res_weights_path):
            raise FileNotFoundError(f"ResNet 가중치 파일이 없습니다: {res_weights_path}")
        self.res_model.load_weights(res_weights_path)
        self.log("ResNet 모델 로드 완료", "SUCCESS")
        
        # Feature extraction 모델 미리 생성 (성능 최적화)
        try:
            if GPU_AVAILABLE:
                self._feature_models['eff'] = Model(self.eff_model.input, self.eff_model.get_layer('gap').output)
                self._feature_models['res'] = Model(self.res_model.input, self.res_model.get_layer('gap').output)
        except Exception as e:
            self.log(f"Feature extraction 모델 생성 실패: {e} (동적 생성으로 대체)", "WARNING")
            self._feature_models = {}
        
        # XGBoost 모델 로드
        try:
            self.log("XGBoost 모델 로드 중...", "INFO")
            xgb_path = f"{self.model_dir}/xgb_model_{self.timestamp}.joblib"
            if not os.path.exists(xgb_path):
                raise FileNotFoundError(f"XGBoost 파일이 없습니다: {xgb_path}")
            
            with tf.device('/CPU:0'):
                self.xgb_model = joblib.load(xgb_path)
            
            if hasattr(self.xgb_model, '_Booster'):
                if not hasattr(self.xgb_model, 'use_label_encoder'):
                    self.xgb_model.use_label_encoder = False
                if not hasattr(self.xgb_model, 'eval_metric'):
                    self.xgb_model.eval_metric = 'logloss'
            
            self.log("XGBoost 모델 로드 완료", "SUCCESS")
        except Exception as e:
            self.xgb_model = None
            self.log(f"XGBoost 로드 실패: {e} (CNN만 사용)", "WARNING")
    
    def build_optimized_model(self, base_cls, num_classes):
        """GPU 최적화된 모델 구조"""
        base = base_cls(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
        
        x = GlobalAveragePooling2D(name='gap')(base.output)
        x = Dense(256, activation='relu', dtype='float32')(x)
        x = Dropout(0.2)(x)
        out = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)
        
        model = Model(inputs=base.input, outputs=out)
        
        # GPU 최적화 컴파일
        if GPU_AVAILABLE:
            optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, epsilon=1e-7)
            try:
                model.compile(
                    optimizer=optimizer, 
                    loss='categorical_crossentropy', 
                    metrics=['accuracy'],
                    run_eagerly=False
                )
            except Exception as e:
                # XLA 컴파일 실패 시 기본 컴파일
                model.compile(
                    optimizer=optimizer, 
                    loss='categorical_crossentropy', 
                    metrics=['accuracy']
                )
        else:
            optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
            model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
        
        return model
    
    def predict_batch_optimized(self, pil_images):
        """GPU 메모리 최적화된 배치 예측"""
        try:
            if not pil_images:
                return []
            
            # 유효한 이미지만 필터링 (벡터화)
            valid_images = [img for img in pil_images if img is not None]
            if not valid_images:
                return [None] * len(pil_images)
            
            device = '/GPU:0' if GPU_AVAILABLE else '/CPU:0'
            
            with tf.device(device):
                # 메모리 효율적 배치 전처리
                batch_size_actual = min(len(valid_images), BATCH_SIZE)
                
                # 텐서 재사용을 위한 캐시 키
                cache_key = f"batch_{batch_size_actual}_{IMG_SIZE[0]}_{IMG_SIZE[1]}"
                
                # NumPy 배열로 한번에 변환 (메모리 효율적)
                batch_images = np.zeros((batch_size_actual, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.uint8)
                
                for i, pil_image in enumerate(valid_images[:batch_size_actual]):
                    try:
                        # PIL to NumPy 직접 변환 (중간 단계 생략)
                        img_array = np.array(pil_image.resize(IMG_SIZE, Image.Resampling.LANCZOS), dtype=np.uint8)
                        batch_images[i] = img_array
                    except Exception:
                        continue
                
                # TensorFlow 텐서로 변환
                batch_tensor = tf.constant(batch_images, dtype=tf.uint8)
                batch_tensor = tf.cast(batch_tensor, tf.float32)
                
                # 병렬 전처리
                with tf.name_scope('preprocessing'):
                    inp_eff = eff_pre(batch_tensor)
                    inp_res = res_pre(batch_tensor)
                
                # 병렬 예측 (GPU 최적화)
                with tf.name_scope('inference'):
                    if GPU_AVAILABLE:
                        # GPU에서 병렬 실행
                        p_eff = self.eff_model(inp_eff, training=False)
                        p_res = self.res_model(inp_res, training=False)
                    else:
                        p_eff = self.eff_model.predict(inp_eff, batch_size=batch_size_actual, verbose=0)
                        p_res = self.res_model.predict(inp_res, batch_size=batch_size_actual, verbose=0)
                
                # 앙상블 (벡터화 연산)
                p_cnn = (p_eff + p_res) * 0.5
                
                # XGBoost 앙상블 (선택적)
                if self.xgb_model and GPU_AVAILABLE and len(self._feature_models) == 2:
                    try:
                        with tf.name_scope('feature_extraction'):
                            feat_eff = self._feature_models['eff'](inp_eff, training=False)
                            feat_res = self._feature_models['res'](inp_res, training=False)
                        
                        # CPU로 이동하여 XGBoost 실행
                        with tf.device('/CPU:0'):
                            feat_combined = tf.concat([feat_eff, feat_res], axis=1).numpy()
                            p_xgb = self.xgb_model.predict_proba(feat_combined)
                        
                        # GPU로 다시 이동하여 앙상블
                        with tf.device(device):
                            p_xgb_tensor = tf.constant(p_xgb, dtype=tf.float32)
                            ensemble = p_cnn * 0.6 + p_xgb_tensor * 0.4
                    except Exception as e:
                        self.log(f"XGBoost 앙상블 실패: {e}", "WARNING")
                        ensemble = p_cnn
                else:
                    ensemble = p_cnn
                
                # 결과 처리 (완전 벡터화)
                ensemble_np = ensemble.numpy()
                
                # Top-3 인덱스 계산 (벡터화)
                top3_indices_batch = np.argpartition(ensemble_np, -3, axis=1)[:, -3:]
                
                batch_results = []
                for i, (pred, top3_indices) in enumerate(zip(ensemble_np, top3_indices_batch)):
                    # 정렬된 인덱스
                    sorted_indices = top3_indices[np.argsort(pred[top3_indices])][::-1]
                    
                    results = []
                    for j, idx in enumerate(sorted_indices):
                        results.append({
                            'rank': j + 1,
                            'food': self.index_to_label[idx],
                            'confidence': float(pred[idx])
                        })
                    batch_results.append(results)
                
                # 원본 크기에 맞춰 결과 조정
                final_results = batch_results[:]
                while len(final_results) < len(pil_images):
                    final_results.append(None)
                
                # 메모리 즉시 정리
                del batch_images, batch_tensor, inp_eff, inp_res, p_eff, p_res, ensemble_np
                if 'feat_combined' in locals():
                    del feat_combined
                
                return final_results[:len(pil_images)]
                
        except Exception as e:
            self.log(f"배치 예측 오류: {e}", "ERROR")
            # 오류 발생 시 상세 로그
            import traceback
            self.log(f"배치 예측 오류 상세: {traceback.format_exc()}", "ERROR")
            return [None] * len(pil_images)
        finally:
            # GPU 메모리 정리
            if GPU_AVAILABLE:
                try:
                    # 안전한 GPU 메모리 정리
                    if hasattr(tf.config.experimental, 'reset_memory_stats'):
                        tf.config.experimental.reset_memory_stats('GPU:0')
                    gc.collect()
                except Exception as e:
                    pass  # GPU 메모리 정리 실패해도 계속 진행
    
    def download_image(self, url):
        """기존 인터페이스 유지"""
        return self.downloader.download_single(url)
    
    def download_batch(self, urls):
        """기존 인터페이스 유지"""
        return self.downloader.download_batch(urls)
    
    def predict_image(self, pil_image):
        """CPU/메모리 최적화된 단일 예측"""
        try:
            device = '/GPU:0' if GPU_AVAILABLE else '/CPU:0'
            
            with tf.device(device):
                # 메모리 효율적 전처리
                img_array = np.array(pil_image.resize(IMG_SIZE, Image.Resampling.LANCZOS), dtype=np.uint8)
                img_tensor = tf.expand_dims(tf.constant(img_array, dtype=tf.float32), axis=0)
                
                inp_eff = eff_pre(img_tensor)
                inp_res = res_pre(img_tensor)
                
                if GPU_AVAILABLE:
                    p_eff = self.eff_model(inp_eff, training=False)
                    p_res = self.res_model(inp_res, training=False)
                else:
                    p_eff = self.eff_model.predict(inp_eff, verbose=0)
                    p_res = self.res_model.predict(inp_res, verbose=0)
                
                p_cnn = (p_eff + p_res) * 0.5
                
                if self.xgb_model:
                    try:
                        with tf.device('/CPU:0'):
                            if GPU_AVAILABLE and len(self._feature_models) == 2:
                                feat_eff = self._feature_models['eff'](inp_eff, training=False).numpy()
                                feat_res = self._feature_models['res'](inp_res, training=False).numpy()
                            else:
                                # 동적으로 feature extraction 모델 생성
                                feat_eff_model = Model(self.eff_model.input, self.eff_model.get_layer('gap').output)
                                feat_res_model = Model(self.res_model.input, self.res_model.get_layer('gap').output)
                                feat_eff = feat_eff_model.predict(inp_eff, verbose=0)
                                feat_res = feat_res_model.predict(inp_res, verbose=0)
                            
                            feat = np.hstack([feat_eff, feat_res])
                            p_xgb = self.xgb_model.predict_proba(feat)
                        
                        with tf.device(device):
                            ensemble = p_cnn.numpy() * 0.6 + p_xgb * 0.4
                    except Exception as e:
                        self.log(f"XGBoost 예측 실패: {e}", "WARNING")
                        ensemble = p_cnn.numpy()
                else:
                    ensemble = p_cnn.numpy()
                
                ensemble = ensemble.flatten()
                top3_indices = np.argsort(ensemble)[-3:][::-1]
                
                results = []
                for i, idx in enumerate(top3_indices):
                    food_name = self.index_to_label[idx]
                    confidence = float(ensemble[idx])
                    results.append({
                        'rank': i + 1,
                        'food': food_name,
                        'confidence': confidence
                    })
                
                # 메모리 즉시 정리
                del img_array, img_tensor, inp_eff, inp_res, p_eff, p_res, ensemble
                
                return results, "성공"
            
        except Exception as e:
            return None, f"예측 오류: {str(e)[:50]}"
    
    def load_properties_data(self, properties_file):
        """메모리 효율적 속성 데이터 로드"""
        if not properties_file or not os.path.exists(properties_file):
            self.log(f"속성 파일이 없습니다: {properties_file}", "WARNING")
            return
        
        try:
            self.log(f"속성 파일 로드 시작: {properties_file}", "INFO")
            
            # 메모리 효율적 로딩 (청크 단위)
            excel_data = pd.ExcelFile(properties_file, engine='openpyxl')
            self.properties_data = {}
            
            total_menus = 0
            for sheet_name in excel_data.sheet_names:
                try:
                    # 메모리 효율적 읽기
                    df = pd.read_excel(properties_file, sheet_name=sheet_name, engine='openpyxl')
                    
                    # 빈 데이터 정리
                    df = df.dropna(how='all').dropna(axis=1, how='all')
                    
                    if df.empty or len(df.columns) < 2:
                        continue
                    
                    # 인덱스 설정
                    ingredient_col = df.columns[0]
                    df = df.set_index(ingredient_col)
                    
                    menu_names = [col for col in df.columns if not pd.isna(col) and str(col).strip()]
                    
                    if not menu_names:
                        continue
                    
                    # 메모리 효율적 저장 (약한 참조 사용)
                    self.properties_data[sheet_name] = df
                    menu_count = len(menu_names)
                    total_menus += menu_count
                    
                    self.log(f"시트 로드: '{sheet_name}' - {len(df)}개 재료, {menu_count}개 메뉴", "SUCCESS")
                    
                    # 주기적 메모리 정리
                    if total_menus > 1000 and total_menus % 500 == 0:
                        gc.collect()
                    
                except Exception as e:
                    self.log(f"시트 '{sheet_name}' 로드 실패: {e}", "ERROR")
                    continue
            
            if self.properties_data:
                self.log(f"속성 파일 로드 완료: {len(self.properties_data)}개 시트, 총 {total_menus}개 메뉴", "SUCCESS")
            else:
                self.log("속성 파일에서 유효한 데이터를 찾을 수 없음", "ERROR")
        
        except Exception as e:
            self.log(f"속성 파일 로드 실패: {e}", "ERROR")
    
    def get_menu_properties(self, menu_name):
        """캐시 최적화된 메뉴 속성 검색"""
        if not menu_name or not self.properties_data:
            return None
        
        menu_name = str(menu_name).strip()
        
        found_properties = {}
        found_menu_name = None
        
        for sheet_name, df in self.properties_data.items():
            menu_columns = [col for col in df.columns if not pd.isna(col) and str(col).strip()]
            
            # 정확 매칭 우선
            exact_match_col = None
            for col in menu_columns:
                if str(col).strip() == menu_name:
                    exact_match_col = col
                    found_menu_name = menu_name
                    break
            
            if exact_match_col:
                sheet_properties = self._extract_properties(df, exact_match_col)
                if sheet_properties:
                    found_properties[sheet_name] = sheet_properties
                continue
            
            # 부분 매칭
            partial_match_col = None
            for col in menu_columns:
                if menu_name in str(col) or str(col) in menu_name:
                    partial_match_col = col
                    found_menu_name = str(col).strip()
                    break
            
            if partial_match_col:
                sheet_properties = self._extract_properties(df, partial_match_col)
                if sheet_properties:
                    found_properties[sheet_name] = sheet_properties
        
        if found_properties:
            return {
                'menu_name': found_menu_name or menu_name,
                'properties': found_properties
            }
        
        return None
    
    def _extract_properties(self, df, col):
        """속성 추출 최적화"""
        sheet_properties = {}
        for idx, value in df[col].items():
            if pd.notna(value) and str(value).strip():
                ingredient_name = str(idx).strip()
                ingredient_value = str(value).strip()
                if (ingredient_name and ingredient_value and 
                    ingredient_value not in ('nan', '-', '')):
                    sheet_properties[ingredient_name] = ingredient_value
        return sheet_properties

# 메모리 효율적 자원 해제
def cleanup_optimized():
    """최적화된 자원 해제"""
    try:
        # 1단계: 가비지 컬렉션
        collected = gc.collect()
        print(f"[CLEANUP] 가비지 컬렉션: {collected}개 객체 정리")
        
        # 2단계: TensorFlow 세션 정리
        try:
            tf.keras.backend.clear_session()
            print("[CLEANUP] TensorFlow 세션 정리 완료")
        except:
            pass
        
        # 3단계: GPU 메모리 정리
        if GPU_AVAILABLE:
            try:
                if hasattr(tf.config.experimental, 'reset_memory_stats'):
                    tf.config.experimental.reset_memory_stats('GPU:0')
                print("[CLEANUP] GPU 메모리 통계 리셋 완료")
            except Exception as e:
                print(f"[CLEANUP] GPU 메모리 정리 실패: {e}")
        
        # 4단계: 프로세스 메모리 최적화
        try:
            import ctypes
            if hasattr(ctypes, 'windll'):
                ctypes.windll.kernel32.SetProcessWorkingSetSize(-1, -1, -1)
        except:
            pass
        
        print("[CLEANUP] 최적화된 자원 정리 완료")
        
    except Exception as e:
        print(f"[CLEANUP] 자원 정리 중 오류: {e}")

atexit.register(cleanup_optimized)

# 메뉴 분석 데이터 수집기 (메모리 효율적)
class MenuAnalysisCollector:
    def __init__(self):
        self.data = defaultdict(lambda: {
            'total_images': 0,
            'correct_predictions': 0,
            'wrong_predictions': defaultdict(int),
            'confidence_sum': 0.0
        })
    
    def add_prediction(self, menu_name, ai_prediction, confidence, is_correct):
        """예측 결과 추가 (메모리 효율적)"""
        if not menu_name:
            return
            
        analysis = self.data[menu_name]
        analysis['total_images'] += 1
        analysis['confidence_sum'] += confidence
        
        if is_correct:
            analysis['correct_predictions'] += 1
        else:
            analysis['wrong_predictions'][ai_prediction] += 1
    
    def get_analysis_data(self):
        """분석 데이터 반환 후 메모리 정리"""
        result = dict(self.data)
        self.clear()
        return result
    
    def clear(self):
        """메모리 정리"""
        self.data.clear()

class FoodPredictorGUI:
    def __init__(self):
        self._vars_created = False
        self._cleanup_done = False
        
        self.root = tk.Tk()
        
        perf_info = f"최적화모드(배치:{BATCH_SIZE})"
        self.root.title(f"음식 이미지 URL 연속 예측 시스템 - {perf_info}")
        self.root.geometry("1400x900")
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        self.predictor = None
        self.is_running = False
        self.stop_requested = False
        
        self.current_image = None
        self.current_results = None
        self.current_properties = None
        self.current_menu_name = None
        self.user_action = None
        
        self.log_queue = queue.Queue()
        self.ui_update_queue = queue.Queue()
        self.translation_table = {}
        
        self.start_time = None
        self.processed_count = 0
        self.success_count = 0
        
        # 메뉴 분석 수집기
        self.menu_collector = MenuAnalysisCollector()
        
        self.setup_ui()
        self.load_translation_table()
        self.check_queues()
    
    def create_tkinter_vars(self):
        if self._vars_created:
            return
        
        try:
            self.folder_var = tk.StringVar(value="./excel_files")
            self.properties_var = tk.StringVar(value="./menu_properties.xlsx")
            
            self.show_images_var = tk.BooleanVar(value=True)
            self.auto_continue_var = tk.BooleanVar(value=False)
            self.fast_mode_var = tk.BooleanVar(value=False)
            self.batch_mode_var = tk.BooleanVar(value=True)
            
            self.status_var = tk.StringVar(value="최적화 모드 준비")
            self.total_files_var = tk.StringVar(value="파일: 0")
            self.total_processed_var = tk.StringVar(value="처리: 0")
            self.total_success_var = tk.StringVar(value="성공: 0")
            self.success_rate_var = tk.StringVar(value="성공률: 0%")
            self.elapsed_time_var = tk.StringVar(value="시간: 00:00")
            
            self.auto_scroll_var = tk.BooleanVar(value=True)
            self._vars_created = True
            
        except Exception as e:
            print(f"Tkinter 변수 생성 오류: {e}")
    
    def setup_ui(self):
        """최적화된 UI 설정"""
        try:
            self.create_tkinter_vars()
            
            main_container = tk.Frame(self.root)
            main_container.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
            
            # 제목
            title_frame = tk.Frame(main_container)
            title_frame.pack(fill=tk.X, pady=(0, 10))
            
            gpu_status = "GPU 최적화모드" if GPU_AVAILABLE else "CPU 최적화"
            title_text = f"음식 이미지 URL 연속 예측 시스템 ({gpu_status})"
            title_label = tk.Label(title_frame, text=title_text, 
                                 font=('Arial', 16, 'bold'), fg='#2E86C1')
            title_label.pack()
            
            perf_text = f"성능 최적화: 배치 {BATCH_SIZE} | 워커 {MAX_WORKERS} | 다운로드 {DOWNLOAD_WORKERS}"
            perf_label = tk.Label(title_frame, text=perf_text, 
                                font=('Arial', 10), fg='#27AE60')
            perf_label.pack()
            
            # 상단 영역
            top_frame = tk.Frame(main_container)
            top_frame.pack(fill=tk.X, pady=(0, 10))
            
            # 좌측 설정
            left_config = tk.LabelFrame(top_frame, text="폴더 및 파일 설정", font=('Arial', 11, 'bold'))
            left_config.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 5))
            
            folder_frame = tk.Frame(left_config)
            folder_frame.pack(fill=tk.X, padx=5, pady=3)
            tk.Label(folder_frame, text="Excel 폴더:", font=('Arial', 10)).pack(side=tk.LEFT)
            tk.Entry(folder_frame, textvariable=self.folder_var, width=40, font=('Arial', 9)).pack(side=tk.LEFT, padx=5, fill=tk.X, expand=True)
            tk.Button(folder_frame, text="찾기", command=self.browse_folder, font=('Arial', 9)).pack(side=tk.RIGHT)
            
            prop_frame = tk.Frame(left_config)
            prop_frame.pack(fill=tk.X, padx=5, pady=3)
            tk.Label(prop_frame, text="속성 파일:", font=('Arial', 10)).pack(side=tk.LEFT)
            tk.Entry(prop_frame, textvariable=self.properties_var, width=40, font=('Arial', 9)).pack(side=tk.LEFT, padx=5, fill=tk.X, expand=True)
            tk.Button(prop_frame, text="찾기", command=self.browse_properties, font=('Arial', 9)).pack(side=tk.RIGHT)
            
            options_frame = tk.Frame(left_config)
            options_frame.pack(fill=tk.X, padx=5, pady=3)
            
            tk.Checkbutton(options_frame, text="이미지 미리보기", variable=self.show_images_var, font=('Arial', 9)).pack(side=tk.LEFT)
            tk.Checkbutton(options_frame, text="자동 진행", variable=self.auto_continue_var, font=('Arial', 9)).pack(side=tk.LEFT, padx=10)
            tk.Checkbutton(options_frame, text="고속 처리", variable=self.fast_mode_var, font=('Arial', 9)).pack(side=tk.LEFT)
            tk.Checkbutton(options_frame, text=f"배치 처리 ({BATCH_SIZE})", variable=self.batch_mode_var, font=('Arial', 9)).pack(side=tk.LEFT, padx=10)
            
            # 우측 로그
            log_frame = tk.LabelFrame(top_frame, text="실행 로그", font=('Arial', 11, 'bold'))
            log_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=(5, 0))
            
            log_control = tk.Frame(log_frame)
            log_control.pack(fill=tk.X, padx=3, pady=2)
            
            tk.Button(log_control, text="지우기", command=self.clear_log, font=('Arial', 8)).pack(side=tk.LEFT)
            tk.Button(log_control, text="저장", command=self.save_log, font=('Arial', 8)).pack(side=tk.LEFT, padx=3)
            tk.Checkbutton(log_control, text="자동 스크롤", variable=self.auto_scroll_var, font=('Arial', 8)).pack(side=tk.RIGHT)
            
            self.log_text = scrolledtext.ScrolledText(log_frame, wrap=tk.WORD, font=('Consolas', 8), 
                                                     height=12, state=tk.DISABLED)
            self.log_text.pack(fill=tk.BOTH, expand=True, padx=3, pady=3)
            
            # 컨트롤 버튼
            control_frame = tk.Frame(main_container)
            control_frame.pack(fill=tk.X, pady=(0, 5))
            
            button_left = tk.Frame(control_frame)
            button_left.pack(side=tk.LEFT)
            
            self.start_button = tk.Button(button_left, text="시작", command=self.start_processing, 
                                        font=('Arial', 12, 'bold'), bg='#E74C3C', fg='white', padx=20)
            self.start_button.pack(side=tk.LEFT, padx=5)
            
            self.stop_button = tk.Button(button_left, text="중지", command=self.stop_processing, 
                                       font=('Arial', 12, 'bold'), bg='#dc3545', fg='white', 
                                       state=tk.DISABLED, padx=20)
            self.stop_button.pack(side=tk.LEFT, padx=5)
            
            self.continue_button = tk.Button(button_left, text="계속", command=self.user_continue, 
                                           font=('Arial', 10), bg='#007bff', fg='white', 
                                           state=tk.DISABLED, padx=15)
            self.continue_button.pack(side=tk.LEFT, padx=5)
            
            self.save_button = tk.Button(button_left, text="저장", command=self.user_save, 
                                       font=('Arial', 10), bg='#ffc107', fg='black', 
                                       state=tk.DISABLED, padx=15)
            self.save_button.pack(side=tk.LEFT, padx=5)
            
            # 통계
            stats_frame = tk.Frame(control_frame)
            stats_frame.pack(side=tk.RIGHT)
            
            tk.Label(stats_frame, textvariable=self.total_files_var, font=('Arial', 10), fg='#E74C3C').pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.total_processed_var, font=('Arial', 10), fg='#E74C3C').pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.total_success_var, font=('Arial', 10), fg='#E74C3C').pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.success_rate_var, font=('Arial', 10), fg='#E74C3C').pack(side=tk.LEFT, padx=5)
            tk.Label(stats_frame, textvariable=self.elapsed_time_var, font=('Arial', 10), fg='#E74C3C').pack(side=tk.LEFT, padx=5)
            
            # 진행률
            progress_frame = tk.Frame(main_container)
            progress_frame.pack(fill=tk.X, pady=(0, 5))
            
            tk.Label(progress_frame, text="진행 상태:", font=('Arial', 10)).pack(side=tk.LEFT)
            tk.Label(progress_frame, textvariable=self.status_var, font=('Arial', 10), fg='#E74C3C').pack(side=tk.LEFT, padx=10)
            
            self.progress = ttk.Progressbar(progress_frame, mode='indeterminate')
            self.progress.pack(side=tk.RIGHT, fill=tk.X, expand=True, padx=10)
            
            # 메인 콘텐츠
            content_frame = tk.Frame(main_container)
            content_frame.pack(fill=tk.BOTH, expand=True)
            
            # 좌측 이미지
            image_panel = tk.LabelFrame(content_frame, text="이미지 분석", font=('Arial', 11, 'bold'))
            image_panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 5))
            
            self.image_frame = tk.Frame(image_panel)
            self.image_frame.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
            
            self.image_label = tk.Label(self.image_frame, text="최적화된 이미지 분석을 시작하려면 '시작' 버튼을 눌러주세요", 
                                       font=('Arial', 10), fg='#E74C3C', 
                                       width=50, height=15, relief=tk.SUNKEN, bg='white')
            self.image_label.pack(expand=True, fill=tk.BOTH)
            
            image_info_frame = tk.Frame(image_panel)
            image_info_frame.pack(fill=tk.X, padx=5, pady=5)
            
            url_frame = tk.LabelFrame(image_info_frame, text="이미지 URL", font=('Arial', 9, 'bold'))
            url_frame.pack(fill=tk.X, pady=2)
            
            self.url_text = tk.Text(url_frame, height=2, font=('Arial', 8), state=tk.DISABLED)
            self.url_text.pack(fill=tk.X, padx=3, pady=2)
            
            result_frame = tk.LabelFrame(image_info_frame, text="예측 결과", font=('Arial', 9, 'bold'))
            result_frame.pack(fill=tk.X, pady=2)
            
            self.result_text = tk.Text(result_frame, height=3, font=('Arial', 9), state=tk.DISABLED)
            self.result_text.pack(fill=tk.X, padx=3, pady=2)
            
            # 우측 정보
            info_panel = tk.Frame(content_frame, width=400)
            info_panel.pack(side=tk.RIGHT, fill=tk.Y, padx=(5, 0))
            info_panel.pack_propagate(False)
            
            menu_frame = tk.LabelFrame(info_panel, text="메뉴 정보", font=('Arial', 11, 'bold'))
            menu_frame.pack(fill=tk.X, pady=(0, 5))
            
            self.menu_info_text = tk.Text(menu_frame, height=4, font=('Arial', 10), state=tk.DISABLED)
            self.menu_info_text.pack(fill=tk.X, padx=3, pady=3)
            
            recipe_frame = tk.LabelFrame(info_panel, text="레시피 정보", font=('Arial', 11, 'bold'))
            recipe_frame.pack(fill=tk.BOTH, expand=True)
            
            self.recipe_text = scrolledtext.ScrolledText(recipe_frame, wrap=tk.WORD, font=('Arial', 9), state=tk.DISABLED)
            self.recipe_text.pack(fill=tk.BOTH, expand=True, padx=3, pady=3)
            
            # 시작 메시지
            self.add_log("최적화된 음식 이미지 예측 시스템이 준비되었습니다.", "INFO")
            if GPU_AVAILABLE:
                self.add_log(f"GPU 최적화 모드 활성화: 배치 {BATCH_SIZE}, 워커 {MAX_WORKERS}개", "SUCCESS")
            else:
                self.add_log(f"CPU 최적화 모드: 배치 {BATCH_SIZE}, 워커 {MAX_WORKERS}개", "SUCCESS")
            self.add_log("Excel 폴더와 속성 파일 경로를 확인한 후 '시작' 버튼을 눌러주세요.", "INFO")
            
        except Exception as e:
            print(f"UI 설정 오류: {e}")
    
    def start_processing(self):
        """최적화된 처리 시작"""
        if self.is_running:
            return
        
        folder_path = self.folder_var.get()
        properties_file = self.properties_var.get()
        
        if not os.path.exists(folder_path):
            messagebox.showerror("오류", f"Excel 폴더가 존재하지 않습니다:\n{folder_path}")
            return
        
        if not os.path.exists('models'):
            messagebox.showerror("오류", "models 폴더가 존재하지 않습니다.")
            return
        
        self.is_running = True
        self.stop_requested = False  
        self.user_action = None
        
        self.start_time = time.time()
        self.processed_count = 0
        self.success_count = 0
        
        # 메뉴 분석 수집기 초기화
        self.menu_collector.clear()
        
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        
        self.progress.config(mode='indeterminate')
        self.progress.start()
        
        self.processing_thread = threading.Thread(target=self.run_processing_optimized, daemon=True)
        self.processing_thread.start()
    
    def run_processing_optimized(self):
        """최적화된 처리 실행"""
        try:
            monitor_memory_detailed()
            
            self.add_log("최적화된 예측 모델 로드 중...", "STEP")
            self.predictor = FoodImagePredictor()
            
            self.predictor.load_models()
            monitor_memory_detailed()
            
            properties_file = self.properties_var.get()
            if properties_file and os.path.exists(properties_file):
                self.predictor.load_properties_data(properties_file)
                monitor_memory_detailed()
            
            self.add_log("모델이 성공적으로 로드되었습니다!", "SUCCESS")
            
            folder_path = self.folder_var.get()
            if not os.path.exists(folder_path):
                self.add_log(f"폴더가 존재하지 않습니다: {folder_path}", "ERROR")
                return
            
            excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))
            excel_files = [f for f in excel_files if not os.path.basename(f).startswith('~')]
            
            if not excel_files:
                self.add_log(f"Excel 파일을 찾을 수 없습니다: {folder_path}", "ERROR")
                return
            
            max_files = 100  # 최적화로 더 많은 파일 처리 가능
            if len(excel_files) > max_files:
                self.add_log(f"파일이 많습니다. 처음 {max_files}개를 최적화 처리합니다.", "WARNING")
                excel_files = excel_files[:max_files]
            
            folder_name = os.path.basename(folder_path.rstrip('/\\'))
            result_folder = f"{folder_path}_결과_최적화"
            
            try:
                if not os.path.exists(result_folder):
                    os.makedirs(result_folder)
                    self.add_log(f"최적화 결과 폴더 생성: {result_folder}", "SUCCESS")
                else:
                    self.add_log(f"최적화 결과 폴더 사용: {result_folder}", "INFO")
            except Exception as e:
                self.add_log(f"결과 폴더 생성 실패: {e}", "ERROR")
                return
            
            self.add_log(f"Excel 파일 {len(excel_files)}개 발견 (최적화 배치 처리)", "SUCCESS")
            
            total_processed = 0
            total_success = 0
            start_time = time.time()
            
            for file_idx, excel_file in enumerate(excel_files):
                if self.stop_requested:
                    self.add_log("사용자 중지 요청으로 처리를 중단합니다.", "WARNING")
                    break
                
                # 최적화된 메모리 정리 (더 적은 빈도)
                if file_idx > 0 and file_idx % 10 == 0:
                    cleanup_optimized()
                    time.sleep(0.1)  # 더 짧은 대기
                    monitor_memory_detailed()
                
                file_name = os.path.basename(excel_file)
                result_file_path = os.path.join(result_folder, file_name)
                
                self.add_log(f"최적화 파일 처리 시작 ({file_idx+1}/{len(excel_files)}): {file_name}", "STEP")
                
                try:
                    file_processed, file_success = self.process_excel_file_optimized(excel_file, result_file_path)
                    total_processed += file_processed
                    total_success += file_success
                    
                    self.update_stats_safe(len(excel_files), total_processed, total_success, start_time)
                    
                    self.add_log(f"파일 처리 완료: {file_name} ({file_success}/{file_processed} 성공)", "SUCCESS")
                    
                    # 다운로드 캐시 통계 출력
                    if hasattr(self.predictor, 'downloader'):
                        cache_stats = self.predictor.downloader.get_cache_stats()
                        if cache_stats['hit_rate'] > 0:
                            self.add_log(f"캐시 효율: {cache_stats['hit_rate']:.1f}% (메모리: {cache_stats['memory_usage_mb']:.1f}MB)", "INFO")
                    
                except Exception as e:
                    self.add_log(f"파일 처리 오류 ({file_name}): {str(e)[:100]}", "ERROR")
                    continue
            
            elapsed = time.time() - start_time
            if elapsed > 0 and total_success > 0:
                avg_speed = total_success / elapsed
                self.add_log(f"최적화 처리 완료! 평균 속도: {avg_speed:.1f}개/초", "SUCCESS")
            
            if total_success > 0:
                self.add_log(f"총 {total_success}개 이미지 분석 성공", "SUCCESS")
                self.add_log(f"결과 파일들이 다음 폴더에 저장되었습니다: {result_folder}", "SUCCESS")
            else:
                self.add_log("처리된 이미지가 없습니다.", "WARNING")
        
        except MemoryError as e:
            self.add_log(f"메모리 부족 오류: {e}", "ERROR")
            self.add_log("배치 크기를 줄이거나 시스템을 재시작하세요.", "ERROR")
        
        except Exception as e:
            self.add_log(f"최적화 처리 중 치명적 오류 발생: {str(e)[:200]}", "ERROR")
            # 상세 오류 로그
            import traceback
            error_details = traceback.format_exc()
            self.add_log(f"오류 상세 정보: {error_details}", "ERROR")
            print(f"전체 오류 스택: {error_details}")  # 콘솔에도 출력
        
        finally:
            cleanup_optimized()
            self.root.after(0, self.finish_processing)
    
    def process_excel_file_optimized(self, excel_file, result_file_path):
        """최적화된 Excel 파일 처리 - 기존 기능 유지하면서 성능 향상"""
        try:
            # 한 번만 Excel 파일 읽기 (I/O 최적화)
            excel_data = pd.ExcelFile(excel_file, engine='openpyxl')
            all_sheets_data = {}
            file_processed = 0
            file_success = 0
            
            # 요약 시트 미리 찾기 (성능 최적화)
            summary_sheet_name = None
            summary_sheet_names = ['요약', '요약시트', 'summary', 'Summary', 'SUMMARY']
            for possible_name in summary_sheet_names:
                if possible_name in excel_data.sheet_names:
                    summary_sheet_name = possible_name
                    break
            
            # 기존 요약 시트 로드 (있는 경우만)
            existing_summary = None
            if summary_sheet_name:
                try:
                    existing_summary = pd.read_excel(excel_file, sheet_name=summary_sheet_name, engine='openpyxl')
                    self.add_log(f"기존 요약 시트 발견: '{summary_sheet_name}' ({len(existing_summary)}개 메뉴)", "SUCCESS")
                except Exception as e:
                    self.add_log(f"요약 시트 로드 실패: {e}", "WARNING")
                    existing_summary = None
            
            for sheet_name in excel_data.sheet_names:
                if self.stop_requested:
                    break
                
                # 요약 시트는 나중에 처리하므로 건너뛰기
                if sheet_name == summary_sheet_name:
                    continue
                
                df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl')
                
                # URL 컬럼 찾기 (기존 로직 유지)
                url_col = None
                for col in df.columns:
                    col_str = str(col).lower()
                    if ('url' in col_str or 'URL' in str(col)) and '수' not in col_str and '개수' not in col_str and '갯수' not in col_str:
                        sample_values = df[col].dropna().head(3)
                        if len(sample_values) > 0:
                            url_like_count = 0
                            for val in sample_values:
                                val_str = str(val).strip()
                                if val_str.startswith(('http://', 'https://')) and '.' in val_str:
                                    url_like_count += 1
                            
                            if url_like_count >= len(sample_values) / 2:
                                url_col = col
                                self.add_log(f"URL 컬럼 발견: '{col}' (샘플: {url_like_count}/{len(sample_values)}개 URL 형태)", "SUCCESS")
                                break
                
                if url_col is None:
                    self.add_log(f"URL 컬럼을 찾을 수 없습니다: {sheet_name}", "WARNING")
                    all_sheets_data[sheet_name] = df
                    continue
                
                valid_urls = df[url_col].notna().sum()
                if valid_urls == 0:
                    all_sheets_data[sheet_name] = df
                    continue
                
                self.add_log(f"최적화 시트 처리: {sheet_name} ({valid_urls}개 URL)", "STEP")
                
                # 결과 컬럼 추가 (기존 로직 유지)
                result_columns = [
                    '예측_음식명', '예측_신뢰도', '예측_2위', '예측_3위',
                    '예측_영어명', '예측_영어_2위', '예측_영어_3위',
                    '레시피_검색키워드', '레시피_매칭결과', '레시피_시트수'
                ]
                
                for col in result_columns:
                    if col not in df.columns:
                        df[col] = ""
                
                # 시트별 처리 (기존 로직 유지하면서 성능 최적화)
                if self.batch_mode_var.get() and valid_urls > 3:
                    sheet_processed, sheet_success = self._process_sheet_batch_optimized(df, url_col, sheet_name, valid_urls)
                else:
                    sheet_processed, sheet_success = self._process_sheet_single_optimized(df, url_col, sheet_name, valid_urls)
                
                file_processed += sheet_processed
                file_success += sheet_success
                
                all_sheets_data[sheet_name] = df
                self.add_log(f"시트 '{sheet_name}' 처리 완료: {sheet_success}/{sheet_processed} 성공", "SUCCESS")
            
            # 요약 시트 처리 (성능에 영향 주지 않도록 최적화)
            if existing_summary is not None:
                try:
                    enhanced_summary = self.enhance_summary_sheet_optimized(existing_summary)
                    if enhanced_summary is not None:
                        all_sheets_data[summary_sheet_name] = enhanced_summary
                        self.add_log(f"요약 시트 AI 분석 추가 완료", "SUCCESS")
                except Exception as e:
                    # 요약 처리 실패해도 메인 처리에는 영향 없음
                    self.add_log(f"요약 시트 처리 오류 (무시함): {e}", "WARNING")
                    all_sheets_data[summary_sheet_name] = existing_summary
            
            # 결과 파일 저장 (I/O 최적화)
            try:
                # ExcelWriter 옵션 안전화
                writer_options = {}
                try:
                    writer_options = {'remove_timezone': True}
                except:
                    pass
                
                with pd.ExcelWriter(result_file_path, engine='openpyxl', **writer_options) as writer:
                    for sheet_name, sheet_df in all_sheets_data.items():
                        # 메모리 효율적 저장
                        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
                
                self.add_log(f"최적화 결과 파일 저장 완료: {os.path.basename(excel_file)}", "SUCCESS")
            except Exception as e:
                self.add_log(f"파일 저장 오류: {e}", "ERROR")
                return 0, 0
            
            return file_processed, file_success
            
        except Exception as e:
            self.add_log(f"Excel 파일 처리 오류: {e}", "ERROR")
            return 0, 0
        finally:
            # 즉시 메모리 정리
            if 'all_sheets_data' in locals():
                del all_sheets_data
            if 'existing_summary' in locals():
                del existing_summary
            gc.collect()
    
    def enhance_summary_sheet_optimized(self, summary_df):
        """성능 영향 없이 요약 시트 향상"""
        try:
            # 메뉴 분석 데이터 가져오기 (메모리 효율적)
            menu_analysis_data = self.menu_collector.get_analysis_data()
            
            if not menu_analysis_data:
                return summary_df  # 분석 데이터가 없으면 원본 반환
            
            # 복사본 생성
            enhanced_summary = summary_df.copy()
            
            # AI 분석 컬럼 추가 (성능 최적화)
            ai_columns = [
                'AI_전체이미지수', 'AI_정확예측수', 'AI_정확률(%)',
                'AI_오예측1위', 'AI_오예측1위_빈도',
                'AI_오예측2위', 'AI_오예측2위_빈도', 
                'AI_오예측3위', 'AI_오예측3위_빈도',
                'AI_평균신뢰도'
            ]
            
            for col in ai_columns:
                if col not in enhanced_summary.columns:
                    enhanced_summary[col] = ""
            
            # 메뉴명 컬럼 찾기 (벡터화 연산)
            menu_col = None
            for col in enhanced_summary.columns:
                if '메뉴' in str(col) or 'menu' in str(col).lower():
                    menu_col = col
                    break
            
            if menu_col is None:
                return enhanced_summary  # 메뉴 컬럼이 없으면 원본 반환
            
            # 벡터화된 매칭 및 업데이트
            matched_count = 0
            for idx, row in enhanced_summary.iterrows():
                menu_name = str(row[menu_col]).strip() if pd.notna(row[menu_col]) else ""
                
                if menu_name and menu_name in menu_analysis_data:
                    analysis = menu_analysis_data[menu_name]
                    matched_count += 1
                    
                    total_images = analysis['total_images']
                    correct_predictions = analysis['correct_predictions']
                    accuracy = (correct_predictions / total_images * 100) if total_images > 0 else 0
                    avg_confidence = (analysis['confidence_sum'] / total_images) if total_images > 0 else 0
                    
                    # 벡터화된 업데이트
                    updates = {
                        'AI_전체이미지수': total_images,
                        'AI_정확예측수': correct_predictions,
                        'AI_정확률(%)': f"{accuracy:.1f}%",
                        'AI_평균신뢰도': f"{avg_confidence:.3f}"
                    }
                    
                    # 오예측 통계 (최적화)
                    wrong_predictions = dict(analysis['wrong_predictions'])
                    if wrong_predictions:
                        sorted_wrong = sorted(wrong_predictions.items(), key=lambda x: x[1], reverse=True)
                        
                        for i, (wrong_pred, count) in enumerate(sorted_wrong[:3]):
                            if i == 0:
                                updates['AI_오예측1위'] = wrong_pred
                                updates['AI_오예측1위_빈도'] = count
                            elif i == 1:
                                updates['AI_오예측2위'] = wrong_pred
                                updates['AI_오예측2위_빈도'] = count
                            elif i == 2:
                                updates['AI_오예측3위'] = wrong_pred
                                updates['AI_오예측3위_빈도'] = count
                    else:
                        updates['AI_오예측1위'] = "(정확예측)"
                        updates['AI_오예측1위_빈도'] = 0
                    
                    # 한 번에 업데이트 (성능 최적화)
                    for col, value in updates.items():
                        enhanced_summary.loc[idx, col] = value
                
                else:
                    # 매칭되지 않은 메뉴 (기본값 설정)
                    default_updates = {
                        'AI_전체이미지수': 0,
                        'AI_정확예측수': 0,
                        'AI_정확률(%)': "0.0%",
                        'AI_평균신뢰도': "0.000",
                        'AI_오예측1위': "분석없음",
                        'AI_오예측1위_빈도': 0
                    }
                    
                    for col, value in default_updates.items():
                        enhanced_summary.loc[idx, col] = value
            
            self.add_log(f"요약 시트 AI 분석 완료: {matched_count}/{len(enhanced_summary)} 메뉴 매칭됨", "SUCCESS")
            return enhanced_summary
            
        except Exception as e:
            self.add_log(f"요약 시트 향상 오류: {e}", "ERROR")
            return summary_df  # 오류 시 원본 반환
    
    def _process_sheet_batch_optimized(self, df, url_col, sheet_name, valid_urls):
        """최적화된 배치 처리 (기존 기능 유지)"""
        sheet_processed = 0
        sheet_success = 0
        
        try:
            # URL 추출 (벡터화 연산)
            url_rows = []
            for idx, row in df.iterrows():
                url = row[url_col]
                if pd.notna(url) and str(url).strip():
                    url_rows.append((idx, str(url).strip()))
            
            if not url_rows:
                return 0, 0
            
            # 동적 배치 크기 조정
            batch_size = min(BATCH_SIZE, len(url_rows))
            total_batches = (len(url_rows) + batch_size - 1) // batch_size
            
            self.add_log(f"최적화 배치 처리 시작: {len(url_rows)}개 URL, {total_batches}개 배치", "INFO")
            
            for batch_idx in range(total_batches):
                if self.stop_requested:
                    break
                
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, len(url_rows))
                batch_url_rows = url_rows[start_idx:end_idx]
                
                # 최적화된 배치 다운로드
                urls = [url for _, url in batch_url_rows]
                download_results = self.predictor.download_batch(urls)
                
                # 이미지 수집 (메모리 효율적)
                batch_images = []
                batch_indices = []
                batch_urls_success = []
                
                for i, (url, (pil_image, msg)) in enumerate(download_results):
                    if i < len(batch_url_rows):
                        row_idx = batch_url_rows[i][0]
                        if pil_image is not None:
                            batch_images.append(pil_image)
                            batch_indices.append(row_idx)
                            batch_urls_success.append(url)
                        else:
                            df.loc[row_idx, '예측_음식명'] = f"오류: {msg}"
                            sheet_processed += 1
                
                # 최적화된 배치 예측
                if batch_images:
                    prediction_results = self.predictor.predict_batch_optimized(batch_images)
                    
                    for i, (results, row_idx, url) in enumerate(zip(prediction_results, batch_indices, batch_urls_success)):
                        sheet_processed += 1
                        
                        if results and len(results) > 0:
                            sheet_success += 1
                            
                            # 결과 저장 (기존 로직 유지)
                            self._save_prediction_results(df, row_idx, results)
                            self._search_and_save_recipe(df, row_idx, results)
                            
                            # 메뉴별 분석 데이터 수집 (성능 최적화)
                            self._collect_menu_analysis_optimized(df, row_idx, results)
                            
                            # UI 업데이트 (첫 번째 이미지만, 성능 최적화)
                            if i == 0:
                                menu_name = self._get_menu_name(df, row_idx)
                                properties_info, search_result_info = self._get_recipe_info(results)
                                
                                self.safe_ui_update(self.update_image_display, 
                                                  batch_images[i], url, results, menu_name, 
                                                  properties_info, search_result_info)
                        else:
                            df.loc[row_idx, '예측_음식명'] = "예측실패"
                
                # 진행률 업데이트 (최적화)
                progress = (batch_idx + 1) / total_batches * 100
                self.safe_ui_update(lambda p: self.progress.config(mode='determinate', value=p), progress)
                
                progress_info = f"최적화배치 {batch_idx + 1}/{total_batches} ({len(batch_images)}개 성공)"
                self.safe_ui_update(lambda info: self.status_var.set(f"처리 중: {info}"), progress_info)
                
                # 성능 추적 업데이트
                self.processed_count += len(batch_url_rows)
                self.success_count += len(batch_images)
                
                success_count = len(batch_images) if batch_images else 0
                self.add_log(f"최적화배치 {batch_idx + 1}/{total_batches} 완료: {success_count}/{len(batch_url_rows)} 성공", "PROCESS")
                
                # 최적화된 메모리 정리 (더 적은 빈도)
                if batch_idx % 10 == 0 and batch_idx > 0:
                    gc.collect()
                
                # 배치 이미지 즉시 해제
                del batch_images
            
            return sheet_processed, sheet_success
            
        except Exception as e:
            self.add_log(f"최적화 배치 처리 오류: {e}", "ERROR")
            return sheet_processed, sheet_success
    
    def _process_sheet_single_optimized(self, df, url_col, sheet_name, valid_urls):
        """최적화된 단일 처리 (기존 기능 유지)"""
        sheet_processed = 0
        sheet_success = 0
        
        for idx, row in df.iterrows():
            if self.stop_requested:
                break
            
            url = row[url_col]
            if pd.isna(url) or str(url).strip() == '':
                continue
            
            sheet_menu_name = self._get_menu_name(df, idx)
            sheet_processed += 1
            
            # 진행률 업데이트 (최적화)
            progress_info = f"[{sheet_processed}/{valid_urls}] {sheet_name}"
            self.safe_ui_update(lambda info: self.status_var.set(f"처리 중: {info}"), progress_info)
            
            progress_percent = (sheet_processed / valid_urls) * 100
            self.safe_ui_update(lambda p: self.progress.config(mode='determinate', value=p), progress_percent)
            
            self.add_log(f"[{sheet_processed:3d}/{valid_urls}] 최적화 이미지 분석 중...", "PROCESS")
            if sheet_menu_name:
                self.add_log(f"    시트 메뉴명: {sheet_menu_name}", "PROCESS")
            
            # 최적화된 이미지 다운로드
            pil_image, download_msg = self.predictor.download_image(url)
            
            if pil_image is None:
                self.add_log(f"다운로드 실패: {download_msg}", "WARNING")
                df.loc[idx, '예측_음식명'] = f"오류: {download_msg}"
                continue
            
            # 최적화된 예측
            results, predict_msg = self.predictor.predict_image(pil_image)
            
            if results is None:
                self.add_log(f"예측 실패: {predict_msg}", "WARNING")
                df.loc[idx, '예측_음식명'] = f"예측오류: {predict_msg}"
                continue
            
            sheet_success += 1
            
            # 결과 저장 (기존 로직 유지)
            self._save_prediction_results(df, idx, results)
            self._search_and_save_recipe(df, idx, results)
            
            # 메뉴별 분석 데이터 수집 (성능 최적화)
            self._collect_menu_analysis_optimized(df, idx, results)
            
            english_prediction = results[0]['food']
            korean_prediction = self.translate_food_name(english_prediction)
            
            self.add_log(f"최적화 예측 성공: {korean_prediction} ({english_prediction}) (신뢰도: {results[0]['confidence']:.3f})", "SUCCESS")
            
            # UI 업데이트 (기존 로직 유지)
            properties_info, search_result_info = self._get_recipe_info(results)
            
            self.current_image = pil_image
            self.current_results = results
            self.current_properties = properties_info
            self.current_menu_name = sheet_menu_name
            self.current_search_info = search_result_info
            
            self.safe_ui_update(self.update_image_display, pil_image, url, results, sheet_menu_name, properties_info, search_result_info)
            
            # 성능 추적 업데이트
            self.processed_count += 1
            self.success_count += 1
            
            # 사용자 입력 대기 (기존 로직 유지)
            if self.show_images_var.get() and not self.fast_mode_var.get():
                user_action = self.wait_for_user_action()
                
                if user_action == 'quit':
                    self.stop_requested = True
                    break
                elif user_action == 'save':
                    self.add_log("이미지 저장됨", "INFO")
            
            # 최적화된 메모리 정리 (더 적은 빈도)
            if sheet_processed % 20 == 0:
                gc.collect()
        
        return sheet_processed, sheet_success
    
    def _collect_menu_analysis_optimized(self, df, idx, results):
        """최적화된 메뉴별 분석 데이터 수집 (성능 영향 최소화)"""
        try:
            # 시트에서 메뉴명 추출
            menu_name = self._get_menu_name(df, idx)
            if not menu_name:
                return
            
            # AI 예측 결과
            ai_prediction = self.translate_food_name(results[0]['food'])
            confidence = results[0]['confidence']
            
            # 메뉴명과 AI 예측 비교 (정확 예측 여부)
            is_correct = (
                menu_name.lower() == ai_prediction.lower() or 
                menu_name.lower() in ai_prediction.lower() or 
                ai_prediction.lower() in menu_name.lower()
            )
            
            # 메모리 효율적 수집
            self.menu_collector.add_prediction(menu_name, ai_prediction, confidence, is_correct)
                    
        except Exception as e:
            # 분석 수집 실패해도 메인 처리에는 영향 없음
            pass
    
    def _get_menu_name(self, df, idx):
        """메뉴명 추출 (캐시 최적화)"""
        for col in df.columns:
            if '메뉴' in str(col) or 'menu' in str(col).lower():
                value = df.loc[idx, col]
                if pd.notna(value):
                    return str(value).strip()
        return None
    
    def _get_recipe_info(self, results):
        """레시피 정보 가져오기 (캐시 최적화)"""
        search_keywords = self.get_recipe_search_keyword(results, None)
        properties_info = None
        search_result_info = None
        
        if search_keywords:
            keyword_info = search_keywords[0]
            keyword = keyword_info['keyword']
            source = keyword_info['source']
            
            properties_info = self.predictor.get_menu_properties(keyword)
            
            if properties_info:
                search_result_info = {
                    'used_keyword': keyword,
                    'keyword_source': source,
                    'found_menu': properties_info['menu_name'],
                    'found_sheets': list(properties_info['properties'].keys())
                }
        
        return properties_info, search_result_info
    
    def _save_prediction_results(self, df, idx, results):
        """예측 결과 저장 (기존 로직 유지)"""
        english_prediction = results[0]['food']
        korean_prediction = self.translate_food_name(english_prediction)
        
        df.loc[idx, '예측_음식명'] = korean_prediction
        df.loc[idx, '예측_신뢰도'] = round(results[0]['confidence'], 3)
        if len(results) > 1:
            df.loc[idx, '예측_2위'] = self.translate_food_name(results[1]['food'])
        if len(results) > 2:
            df.loc[idx, '예측_3위'] = self.translate_food_name(results[2]['food'])
        
        df.loc[idx, '예측_영어명'] = english_prediction
        if len(results) > 1:
            df.loc[idx, '예측_영어_2위'] = results[1]['food']
        if len(results) > 2:
            df.loc[idx, '예측_영어_3위'] = results[2]['food']
    
    def _search_and_save_recipe(self, df, idx, results):
        """레시피 검색 및 저장 (기존 로직 유지)"""
        search_keywords = self.get_recipe_search_keyword(results, None)
        
        if search_keywords:
            keyword_info = search_keywords[0]
            keyword = keyword_info['keyword']
            source = keyword_info['source']
            
            df.loc[idx, '레시피_검색키워드'] = f"{keyword} ({source})"
            
            properties_info = self.predictor.get_menu_properties(keyword)
            
            if properties_info:
                df.loc[idx, '레시피_매칭결과'] = properties_info['menu_name']
                df.loc[idx, '레시피_시트수'] = len(properties_info['properties'])
            else:
                df.loc[idx, '레시피_매칭결과'] = "매칭실패"
                df.loc[idx, '레시피_시트수'] = 0
        else:
            df.loc[idx, '레시피_검색키워드'] = "키워드생성실패"
            df.loc[idx, '레시피_매칭결과'] = "검색실패"
            df.loc[idx, '레시피_시트수'] = 0
    
    def on_closing(self):
        if self._cleanup_done:
            return
        
        try:
            self._cleanup_done = True
            self.stop_requested = True
            self.is_running = False
            
            if hasattr(self, 'processing_thread') and self.processing_thread and self.processing_thread.is_alive():
                self.processing_thread.join(timeout=2)
            
            if hasattr(self, 'image_label') and hasattr(self.image_label, 'image'):
                self.image_label.image = None
            
            self.current_image = None
            
            if self.predictor:
                self.predictor = None
            
            # 큐 정리
            try:
                while not self.log_queue.empty():
                    self.log_queue.get_nowait()
                while not self.ui_update_queue.empty():
                    self.ui_update_queue.get_nowait()
            except:
                pass
            
            # 메뉴 수집기 정리
            if hasattr(self, 'menu_collector'):
                self.menu_collector.clear()
            
            cleanup_optimized()
            self.root.quit()
            self.root.destroy()
            
        except Exception as e:
            print(f"종료 중 오류: {e}")
            try:
                self.root.destroy()
            except:
                pass
    
    def check_queues(self):
        if self._cleanup_done or self.stop_requested:
            return
        
        try:
            # 로그 큐 처리 (배치 처리로 성능 향상)
            log_messages = []
            while not self.log_queue.empty() and len(log_messages) < 10:
                try:
                    message, level = self.log_queue.get_nowait()
                    log_messages.append((message, level))
                except queue.Empty:
                    break
                except:
                    break
            
            # 배치로 로그 처리
            for message, level in log_messages:
                self._add_log_to_ui(message, level)
            
            # UI 업데이트 큐 처리
            ui_updates = []
            while not self.ui_update_queue.empty() and len(ui_updates) < 5:
                try:
                    update_func, args = self.ui_update_queue.get_nowait()
                    ui_updates.append((update_func, args))
                except queue.Empty:
                    break
                except:
                    break
            
            # 배치로 UI 업데이트
            for update_func, args in ui_updates:
                try:
                    update_func(*args)
                except:
                    pass
            
            if not self.stop_requested and not self._cleanup_done:
                self.root.after(50, self.check_queues)
                
        except Exception as e:
            print(f"큐 확인 중 오류: {e}")
    
    def add_log(self, message, level="INFO"):
        if self._cleanup_done:
            return
        
        try:
            if threading.current_thread() == threading.main_thread():
                self._add_log_to_ui(message, level)
            else:
                self.log_queue.put((message, level))
        except Exception as e:
            print(f"로그 추가 오류: {e}")
    
    def _add_log_to_ui(self, message, level="INFO"):
        if self._cleanup_done:
            return
        
        try:
            if not hasattr(self, 'log_text') or not self.log_text.winfo_exists():
                return
            
            timestamp = datetime.now().strftime("%H:%M:%S")
            
            color_map = {
                "INFO": "#000000",
                "SUCCESS": "#27AE60",
                "WARNING": "#F39C12",
                "ERROR": "#E74C3C",
                "STEP": "#3498DB",
                "PROCESS": "#9B59B6"
            }
            
            color = color_map.get(level, "#000000")
            
            self.log_text.config(state=tk.NORMAL)
            self.log_text.insert(tk.END, f"[{timestamp}] [{level}] {message}\n")
            
            # 최적화된 태그 처리
            line_start = self.log_text.index("end-2l linestart")  
            line_end = self.log_text.index("end-1l lineend")
            tag_name = f"{level}_{timestamp}_{hash(message) % 10000}"
            self.log_text.tag_add(tag_name, line_start, line_end)
            self.log_text.tag_config(tag_name, foreground=color)
            
            self.log_text.config(state=tk.DISABLED)
            
            if hasattr(self, 'auto_scroll_var') and self.auto_scroll_var.get():
                self.log_text.see(tk.END)
            
        except Exception as e:
            print(f"UI 로그 업데이트 오류: {e}")
    
    def clear_log(self):
        try:
            self.log_text.config(state=tk.NORMAL)
            self.log_text.delete(1.0, tk.END)
            self.log_text.config(state=tk.DISABLED)
        except Exception as e:
            print(f"로그 지우기 오류: {e}")
    
    def save_log(self):
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"optimized_log_{timestamp}.txt"
            
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(self.log_text.get(1.0, tk.END))
            
            self.add_log(f"로그가 저장되었습니다: {filename}", "SUCCESS")
            
        except Exception as e:
            self.add_log(f"로그 저장 실패: {e}", "ERROR")
    
    def browse_folder(self):
        try:
            folder = filedialog.askdirectory(title="Excel 파일이 있는 폴더를 선택하세요")
            if folder:
                self.folder_var.set(folder)
        except Exception as e:
            self.add_log(f"폴더 선택 오류: {e}", "ERROR")
    
    def browse_properties(self):
        try:
            file = filedialog.askopenfilename(
                title="메뉴 속성 Excel 파일을 선택하세요",
                filetypes=[("Excel files", "*.xlsx *.xls"), ("All files", "*.*")]
            )
            if file:
                self.properties_var.set(file)
        except Exception as e:
            self.add_log(f"속성 파일 선택 오류: {e}", "ERROR")
    
    def load_translation_table(self):
        try:
            self.translation_table = {
                'BBQ': '바비큐립', 'baguette': '바게트', 'banh_mi': '반미', 'bingsu': '팥빙수',
                'bulgogi': '불고기', 'bunza': '분짜', 'burger': '치즈버거', 'burrito': '부리또',
                'cake': '치즈케이크', 'chicken': '후라이드치킨', 'cookie': '쿠키', 'croissant': '크루아상',
                'croque_monsieur': '크로크무슈', 'curry': '레드커리', 'dim_sum': '딤섬', 
                'egg_benedict': '에그베네딕트', 'french_fries': '감자튀김', 'french_toast': '프렌치토스트',
                'galbi': '돼지갈비', 'gimbap': '참치김밥', 'gratin': '그라탱', 'hot_pot': '훠궈',
                'jajangmyeon': '짜장면', 'japchae': '잡채밥', 'kebap': '케밥', 'kimchi_stew': '김치찌개',
                'korean_pancake': '해물전', 'lasana': '라자냐', 'macaroon': '마카롱', 'mapa_tofu': '마파두부밥',
                'muffin': '머핀', 'nachos': '나초', 'pad_thai': '팟타이', 'pan_cake': '팬케이크',
                'pasta': '마라파스타', 'pizza': '한국식피자', 'quesadilla': '케사디아', 'ramen': '라멘',
                'rice_noodle': '쌀국수', 'risotto': '해물리조또', 'salad': '시저샐러드', 'sashimi': '생선회',
                'seaweed_soup': '미역국', 'soba': '소바', 'soup': '크림수프', 'steak': '스테이크와감자',
                'sushi': '연어초밥', 'takoyaki': '타코야키', 'tteokbokki': '국물떡볶이', 'udon': '우동'
            }
            
            self.add_log(f"매핑 테이블 로드 완료: {len(self.translation_table)}개", "SUCCESS")
            
        except Exception as e:
            self.add_log(f"번역 테이블 로드 실패: {e}", "ERROR")
    
    def translate_food_name(self, english_name):
        return self.translation_table.get(english_name, english_name)
    
    def get_recipe_search_keyword(self, results, sheet_menu_name):
        search_keywords = []
        
        if results:
            english_prediction = results[0]['food']
            
            korean_mapped = self.translation_table.get(english_prediction)
            if korean_mapped:
                search_keywords.append({
                    'keyword': korean_mapped,
                    'source': 'AI예측(한글매핑)',
                    'confidence': results[0]['confidence']
                })
            else:
                search_keywords.append({
                    'keyword': english_prediction,
                    'source': 'AI예측(매핑없음)',
                    'confidence': results[0]['confidence']
                })
        
        return search_keywords
    
    def stop_processing(self):
        self.stop_requested = True
        self.add_log("중지 요청됨 - 현재 배치 완료 후 중단됩니다.", "WARNING")
    
    def user_continue(self):
        self.user_action = 'continue'
    
    def user_save(self):
        self.user_action = 'save'
    
    def wait_for_user_action(self):
        if self.auto_continue_var.get():
            return 'continue'
        
        self.continue_button.config(state=tk.NORMAL)
        self.save_button.config(state=tk.NORMAL)
        
        self.user_action = None
        while self.user_action is None and not self.stop_requested:
            time.sleep(0.01)  # 더 빠른 응답
        
        self.continue_button.config(state=tk.DISABLED)
        self.save_button.config(state=tk.DISABLED)
        
        return self.user_action or 'quit'
    
    def safe_ui_update(self, update_func, *args):
        try:
            if threading.current_thread() == threading.main_thread():
                update_func(*args)
            else:
                self.ui_update_queue.put((update_func, args))
        except Exception as e:
            print(f"UI 업데이트 오류: {e}")
    
    def update_stats_safe(self, total_files=0, total_processed=0, total_success=0, start_time=None):
        def _update():
            try:
                if hasattr(self, 'total_files_var'):
                    self.total_files_var.set(f"파일: {total_files}")
                if hasattr(self, 'total_processed_var'):
                    self.total_processed_var.set(f"처리: {total_processed}")
                if hasattr(self, 'total_success_var'):
                    self.total_success_var.set(f"성공: {total_success}")
                
                if total_processed > 0:
                    success_rate = (total_success / total_processed) * 100
                    if hasattr(self, 'success_rate_var'):
                        self.success_rate_var.set(f"성공률: {success_rate:.1f}%")
                else:
                    if hasattr(self, 'success_rate_var'):
                        self.success_rate_var.set("성공률: 0%")
                
                if start_time and hasattr(self, 'elapsed_time_var'):
                    elapsed = time.time() - start_time
                    hours = int(elapsed // 3600)
                    minutes = int((elapsed % 3600) // 60)
                    seconds = int(elapsed % 60)
                    
                    throughput = total_success / elapsed if elapsed > 0 else 0
                    
                    if hours > 0:
                        time_str = f"시간: {hours:02d}:{minutes:02d}:{seconds:02d} ({throughput:.1f}/s)"
                    else:
                        time_str = f"시간: {minutes:02d}:{seconds:02d} ({throughput:.1f}/s)"
                    self.elapsed_time_var.set(time_str)
            except Exception as e:
                print(f"통계 업데이트 오류: {e}")
        
        self.safe_ui_update(_update)
    
    def update_image_display(self, pil_image, url, results, menu_name, properties_info, search_result_info=None):
        """최적화된 이미지 표시 업데이트"""
        try:
            if not pil_image or self._cleanup_done:
                return
            
            # 이미지 리사이즈 최적화
            display_size = (400, 300)
            img_copy = pil_image.copy()
            img_copy.thumbnail(display_size, Image.Resampling.LANCZOS)
            
            tk_image = ImageTk.PhotoImage(img_copy)
            
            if hasattr(self, 'image_label') and self.image_label.winfo_exists():
                self.image_label.config(image=tk_image, text="")
                self.image_label.image = tk_image
            
            # 텍스트 업데이트 최적화 (배치 처리)
            text_updates = []
            
            if hasattr(self, 'url_text') and self.url_text.winfo_exists():
                text_updates.append((self.url_text, str(url)))
            
            if hasattr(self, 'result_text') and self.result_text.winfo_exists():
                if results:
                    result_content = ""
                    for i, result in enumerate(results[:3]):
                        korean_name = self.translation_table.get(result['food'], result['food'])
                        result_content += f"{i+1}위: {korean_name} ({result['food']}) - 신뢰도: {result['confidence']:.3f}\n"
                    text_updates.append((self.result_text, result_content))
                else:
                    text_updates.append((self.result_text, "예측 결과 없음"))
            
            if hasattr(self, 'menu_info_text') and self.menu_info_text.winfo_exists():
                menu_content = ""
                if menu_name:
                    menu_content += f"시트 메뉴명: {menu_name}\n"
                
                if results:
                    english_prediction = results[0]['food']
                    korean_mapped = self.translation_table.get(english_prediction, english_prediction)
                    menu_content += f"AI 예측: {korean_mapped}\n"
                    menu_content += f"신뢰도: {results[0]['confidence']:.3f}\n"
                    
                    if search_result_info:
                        menu_content += f"레시피 키워드: {search_result_info['used_keyword']} ({search_result_info['keyword_source']})\n"
                        menu_content += f"레시피 매칭: {search_result_info['found_menu']}\n"
                    else:
                        menu_content += f"레시피 매칭: 없음\n"
                
                text_updates.append((self.menu_info_text, menu_content))
            
            if hasattr(self, 'recipe_text') and self.recipe_text.winfo_exists():
                if search_result_info and properties_info:
                    found_menu = properties_info['menu_name']
                    properties = properties_info['properties']
                    used_keyword = search_result_info['used_keyword']
                    keyword_source = search_result_info['keyword_source']
                    
                    recipe_content = f"최적화 레시피 분석 결과\n"
                    recipe_content += f"레시피 기준 메뉴: {found_menu}\n"
                    recipe_content += f"검색 키워드: {used_keyword} ({keyword_source})\n"
                    if found_menu != used_keyword:
                        recipe_content += f"매칭 결과: {used_keyword} -> {found_menu}\n"
                    recipe_content += "\n"
                    
                    sheet_names = {
                        '주요식재료': '주요 식재료',
                        '양념소스': '양념 및 소스', 
                        '육수베이스': '육수/베이스',
                        '조리순서': '조리 순서',
                        '조리정보': '조리 정보',
                        '특징맛': '맛 특성',
                        '영양정보': '영양 정보',
                        '서빙조리팁': '조리 팁'
                    }
                    
                    for sheet_name, sheet_data in properties.items():
                        sheet_title = sheet_names.get(sheet_name, sheet_name)
                        recipe_content += f"\n--- {sheet_title} ---\n"
                        
                        count = 0
                        for ingredient, amount in sheet_data.items():
                            if count >= 15:
                                recipe_content += "  ... (더 많은 항목 있음)\n"
                                break
                            recipe_content += f"  • {ingredient}: {amount}\n"
                            count += 1
                        
                        recipe_content += "\n"
                else:
                    recipe_content = "최적화 레시피 정보를 찾을 수 없습니다.\n\n"
                    
                    if results:
                        english_prediction = results[0]['food']
                        korean_mapped = self.translation_table.get(english_prediction, english_prediction)
                        
                        recipe_content += f"AI 예측: {english_prediction} -> {korean_mapped}\n"
                        
                        if korean_mapped == english_prediction:
                            recipe_content += f"매핑 상태: 매핑테이블에 '{english_prediction}' 없음\n"
                        else:
                            recipe_content += f"매핑 상태: 매핑테이블에서 '{korean_mapped}'로 변환됨\n"
                        
                        recipe_content += f"검색 결과: 속성파일에 '{korean_mapped}' 메뉴 없음\n\n"
                        recipe_content += "해결 방법:\n"
                        recipe_content += "1. 매핑테이블에 해당 영어명 추가\n"
                        recipe_content += "2. 속성파일에 해당 한글 메뉴명 추가"
                    else:
                        recipe_content += "예측 결과 정보가 없습니다."
                
                text_updates.append((self.recipe_text, recipe_content))
            
            # 배치로 텍스트 업데이트 (성능 향상)
            for text_widget, content in text_updates:
                try:
                    text_widget.config(state=tk.NORMAL)
                    text_widget.delete(1.0, tk.END)
                    text_widget.insert(tk.END, content)
                    text_widget.config(state=tk.DISABLED)
                except:
                    pass
            
            # 메모리 즉시 정리
            del img_copy, tk_image
            
        except Exception as e:
            print(f"이미지 표시 업데이트 오류: {e}")
    
    def finish_processing(self):
        try:
            self.is_running = False
            self.start_button.config(state=tk.NORMAL)
            self.stop_button.config(state=tk.DISABLED)
            self.continue_button.config(state=tk.DISABLED)
            self.save_button.config(state=tk.DISABLED)
            self.progress.stop()
            self.progress.config(mode='determinate', value=0)
            
            if self.stop_requested:
                self.status_var.set("사용자 중지")
                self.add_log("사용자 요청으로 최적화 처리가 중단되었습니다.", "WARNING")
            else:
                self.status_var.set("최적화 처리 완료")
                self.add_log("모든 최적화 처리가 완료되었습니다!", "SUCCESS")
                
        except Exception as e:
            print(f"처리 완료 정리 중 오류: {e}")
    
    def run(self):
        try:
            self.root.mainloop()
        except KeyboardInterrupt:
            print("사용자가 프로그램을 중단했습니다.")
        except Exception as e:
            print(f"GUI 실행 중 오류: {e}")
        finally:
            if not self._cleanup_done:
                self.on_closing()

def setup_performance_monitoring():
    """성능 모니터링 설정"""
    print("=== 최적화 모드 활성화 ===")
    print(f"배치 크기: {BATCH_SIZE}")
    print(f"최대 워커: {MAX_WORKERS}")
    print(f"다운로드 워커: {DOWNLOAD_WORKERS}")
    print(f"GPU 사용: {'예' if GPU_AVAILABLE else '아니오'}")
    
    monitor_memory_detailed()
    
    memory_gb = psutil.virtual_memory().total / (1024**3)
    if memory_gb < 8:
        print("WARNING: 메모리가 8GB 미만입니다.")
    
    if BATCH_SIZE > 32:
        print("INFO: 큰 배치 크기로 고속 처리됩니다.")
    
    if MAX_WORKERS > 16:
        print("INFO: 많은 워커로 병렬 처리됩니다.")
    
    print("시스템이 최적화되어 구동됩니다.")

def main():
    """최적화된 메인 함수"""
    try:
        setup_performance_monitoring()
        
        profiler = PerformanceProfiler()
        profiler.start()
        
        app = FoodPredictorGUI()
        
        try:
            import signal
            def signal_handler(signum, frame):
                print(f"시그널 {signum} 수신 - 최적화 모드 안전한 종료를 시도합니다.")
                app.on_closing()
            
            signal.signal(signal.SIGINT, signal_handler)
            signal.signal(signal.SIGTERM, signal_handler)
        except:
            pass
        
        profiler.checkpoint("GUI 초기화")
        
        app.run()
        
        profiler.checkpoint("프로그램 종료")
        profiler.summary()
        
    except KeyboardInterrupt:
        print("사용자가 프로그램을 중단했습니다.")
    except Exception as e:
        print(f"최적화 모드 실행 중 오류: {e}")
        import traceback
        traceback.print_exc()
    finally:
        try:
            cleanup_optimized()
            print("최적화 모드 종료 완료")
        except:
            pass

if __name__ == '__main__':
    main()

[OPTIMIZE] CPU: 16코어, RAM: 19.7GB, GPU: 1개
[OPTIMIZE] 최적화 설정 - 배치: 23, 워커: 32, 다운로드: 16
[OPTIMIZE] 예상 메모리 사용량: ~5.8GB
[GPU OPTIMIZE] 혼합정밀도 활성화
[GPU OPTIMIZE] XLA 컴파일 활성화
[GPU OPTIMIZE] GPU 최적화 완료: 1개
=== 최적화 모드 활성화 ===
배치 크기: 23
최대 워커: 32
다운로드 워커: 16
GPU 사용: 예
[MEMORY] 물리 메모리: 9.2GB / 19.7GB (46.7%)
[MEMORY] 사용 가능: 10.5GB
[GPU MEMORY] 현재: 0.0GB, 최대: 0.0GB
[PROCESS MEMORY] 현재 프로세스: 1.0GB RSS
INFO: 많은 워커로 병렬 처리됩니다.
시스템이 최적화되어 구동됩니다.
[MEMORY] 물리 메모리: 9.2GB / 19.7GB (46.5%)
[MEMORY] 사용 가능: 10.5GB
[GPU MEMORY] 현재: 0.0GB, 최대: 0.0GB
[PROCESS MEMORY] 현재 프로세스: 1.0GB RSS
[12:28:39] [INFO] 모델 로드 중...
[12:28:39] [SUCCESS] GPU 최적화 모드로 모델 로드 중...
[12:28:39] [SUCCESS] 라벨 매핑 로드 완료 - 클래스 수: 84
[12:28:39] [INFO] EfficientNet 모델 로드 중...
[12:28:40] [SUCCESS] EfficientNet 모델 로드 완료
[12:28:40] [INFO] ResNet 모델 로드 중...
[12:28:42] [SUCCESS] ResNet 모델 로드 완료
[12:28:42] [INFO] XGBoost 모델 로드 중...
[12:28:42] [SUCCESS] XGBoost 모델 로드 완료
[MEMORY] 물리 메모리: 9.2GB / 19.7GB (46.9%)
[MEMORY] 사용 가능: 10.5GB
[GPU MEMORY] 현재: 0